## Agent II: Memory Agent - Practice & Exercises

In this notebook, you will practice building a stateful chatbot with conversation memory using LangGraph. You will then complete an exercise to build a Math Tutor chatbot.

### Part 1: Follow-Along Practice

#### Goal
Build a stateful chatbot that remembers conversation history and appends responses back to the message list.

#### Steps:
1. Define the `AgentState` schema containing `messages` (a list of messages, using both HumanMessage and AIMessage).
2. Create the `process` node function that invokes the LLM with the full message history and appends the result to `messages`.
3. Build the graph, compile it, and invoke it sequentially to see it remember previous details.

In [ ]:
from typing import TypedDict, Union
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

# 1. Define the State Schema
class AgentState(TypedDict):
    messages: list[Union[HumanMessage, AIMessage]]

In [ ]:
# 2. Define Node Function
llm = ChatOpenAI(model="gpt-4o")

def process(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    state["messages"].append(AIMessage(content=response.content))
    print(f"\nAI: {response.content}")
    return state

In [ ]:
# 3. Build and Compile
graph = StateGraph(AgentState)
graph.add_node("chatbot", process)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)
app = graph.compile()

In [ ]:
# 4. Invoke sequentially to test memory
state = {"messages": []}
state["messages"].append(HumanMessage(content="Hi, my name is Satish."))
state = app.invoke(state)

state["messages"].append(HumanMessage(content="What is my name?"))
app.invoke(state)

---

### Part 2: Practice Exercises

#### Exercise 1: Math Tutor Memory Agent

##### Goal
Build a Math Tutor chatbot that remembers the user's name and guides them through math questions, keeping conversation context.

##### Sample Input
Sequential invocations:
1. `"Hi, I am Rahul. I need help with arithmetic."`
2. `"What is my name and what subject did I ask for help with?"`

##### Sample Output
1. `"Hello Rahul! I'd be happy to help you with arithmetic. What is your first question?"`
2. `"Your name is Rahul, and you asked for help with arithmetic."`

##### Plan
1. **Define State Schema:** Create a `TypedDict` for the list of conversation messages.
2. **Define Node Function:** Write `tutor_node` that binds a math system message instruction, runs the LLM on the chat list, and appends the response.
3. **Build & Compile Graph:** Register the tutor node, set entry/finish points, and compile the graph.
4. **Invoke Graph:** Test with sequential questions to confirm memory retention.

In [ ]:
# TODO: Define your State, Node, Graph, Compile, and Invoke it here


<details>
<summary>💡 Click to see solution</summary>

```python
from typing import TypedDict, Union
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

## 1. State Schema
class AgentState(TypedDict):
    messages: list[Union[HumanMessage, AIMessage]]

llm = ChatOpenAI(model="gpt-4o")

## 2. Node Function
def tutor_node(state: AgentState) -> AgentState:
    sys = SystemMessage(content="You are an encouraging math tutor bot. Remember the user's name and preferences.")
    response = llm.invoke([sys] + state["messages"])
    state["messages"].append(AIMessage(content=response.content))
    print(f"\nTutor: {response.content}")
    return state

## 3. Create Graph
graph = StateGraph(AgentState)
graph.add_node("tutor", tutor_node)
graph.add_edge(START, "tutor")
graph.add_edge("tutor", END)
app = graph.compile()

## 4. Invoke sequentially
state = {"messages": []}
state["messages"].append(HumanMessage(content="Hi, I am Rahul. I need help with arithmetic."))
state = app.invoke(state)

state["messages"].append(HumanMessage(content="What is my name and what subject did I ask for help with?"))
app.invoke(state)
```
</details>